In [103]:
import pandas as pd
import numpy as np
import functools

In [ ]:
prod_df = pd.read_csv("../data/products.csv")
prod_df.head()

,category,subcategory,name,description,url,price,image
0,accessories,bag,Classic Black Patent Leather Purse,"Introducing the Classic Black Patent Bag, a bl...",/images/Classic_Black_Patent_Leather_Purse1-sm...,49.9,/images/Classic_Black_Patent_Leather_Purse1-sm...
1,accessories,bag,Embroidered Cream Purse with Gold Detail,Introducing the Embroidered Cream purse with g...,/images/Creme_clutch_purse1-small.jpg,69.9,/images/Creme_clutch_purse1-small.jpg
2,accessories,bag,Ratan Basket Shoulder Bag,The Ratan Basket Shoulder Bag is a charming bl...,/images/Ratan_Basket_Shoulder_Bag-small2.jpg,89.9,/images/Ratan_Basket_Shoulder_Bag-small2.jpg
3,accessories,bag,Embroidered Leather Bohemian Purse,"Introducing the Embroidered Fabric Purse, a de...",/images/Embroidered_Leather_Bohemian_Purse-sma...,49.9,/images/Embroidered_Leather_Bohemian_Purse-sma...
4,apparel,dress,Black Polka-Dotted Slip Dress,Discover timeless elegance with our Black Polk...,/images/Black_Polka_Dotted_Slip_Dress.jpeg,59.9,/images/Black_Polka_Dotted_Slip_Dress.jpeg


In [5]:
prod_df.shape

(15, 7)

In [75]:
prod_ext_df = pd.read_csv("../data/products_extended.csv")
prod_ext_df.insert(0, "product_id", range(1, 1 + len(prod_ext_df)))
prod_ext_df.head()

,product_id,category,subcategory,name,description,url,price,image
0,1,jewelry,bracelet,Southwest Bracelet,The Southwest Bracelet combines elegance with ...,/images/Southwest_Bracelet.jpg,169.99,/images/Southwest_Bracelet.jpg
1,2,jewelry,bracelet,Jeweled Bracelet Watch,"Introducing the Jeweled Bracelet Watch, a styl...",/images/Jeweled_Bracelet_Watch.jpg,259.99,/images/Jeweled_Bracelet_Watch.jpg
2,3,jewelry,bracelet,Isis Bracelet,The Isis Bracelet add glamour to any outfit. T...,/images/Isis_Bracelet.jpg,269.99,/images/Isis_Bracelet.jpg
3,4,jewelry,bracelet,Butterfly Mixed Metal Bracelet,Introducing the Butterfly Mixed Metal Bracelet...,/images/Butterfly_Mixed_Metal_Bracelet.jpg,89.99,/images/Butterfly_Mixed_Metal_Bracelet.jpg
4,5,jewelry,bracelet,Butterfly Gold Chain Bracelet,"Introducing the Butterfly Gold Chain Bracelet,...",/images/Butterfly_Gold_Chain_Bracelet.jpg,59.99,/images/Butterfly_Gold_Chain_Bracelet.jpg


In [112]:
prod_ext_df.to_csv("../data/products_extended_w_id.csv", index=False)

In [76]:
prod_ext_df.shape

(218, 8)

In [77]:
prod_ext_df['category'].value_counts()

category
apparel        103
accessories     60
footwear        35
jewelry         20
Name: count, dtype: int64

In [78]:
prod_ext_df['subcategory'].value_counts()

subcategory
skirt                 40
shoes                 35
dress                 33
bag                   31
top blouse sweater    30
sunglasses            29
bracelet               8
earrings               7
necklace               5
Name: count, dtype: int64

In [79]:
import requests
import json

response = requests.post(
    "http://localhost:11434/api/embed",
    json={
        "model": "nomic-embed-text",
        "input": ["Samsung S90F OLED TV", "LG C5 OLED TV"]
    }
)

In [80]:
response_dict = json.loads(response.content)

In [81]:
emb_arr = np.array(response_dict['embeddings'])

In [82]:
def cosine_sim(v1, v2):
    return np.dot(v1, v2) / np.linalg.norm(v1) * np.linalg.norm(v2) 
    

In [83]:
cosine_sim(emb_arr[0, :], emb_arr[1, :])

np.float64(0.7368447046879271)

In [84]:
DATA_FIELDS = [
    'product_id',
    'category',
    'subcategory',
    'name',
    'description',
    'price'
]

In [85]:
row = prod_ext_df.iloc[0]

In [86]:
row.to_dict()

{'product_id': 1,
 'category': 'jewelry',
 'subcategory': 'bracelet',
 'name': 'Southwest Bracelet',
 'description': "The Southwest Bracelet combines elegance with a touch of American Indian glamour. This jeweled bracelet is perfect for making a statement at any occasion, whether you're at a luncheon or an evening out. The bracelet is crafted from high-quality crystals and natural stones, offering a lightweight and comfortable fit. The bracelet is a sparkling accent, catching the light and adding a touch of style.  Pair this accessory with a chic white blouse and a tailored skirt for a polished look, or dress it down with jeans for an elevated everyday style.",
 'url': '/images/Southwest_Bracelet.jpg',
 'price': 169.99,
 'image': '/images/Southwest_Bracelet.jpg'}

In [87]:
for idx, row in prod_ext_df.iterrows():
    rec = row[DATA_FIELDS].to_dict()
    

In [113]:
import weaviate
import weaviate.classes as wvc

In [115]:
# Create collection

with weaviate.connect_to_local() as client:
    products = client.collections.delete("products")

    products = client.collections.create(
        name="products",
        vector_config=wvc.config.Configure.Vectors.text2vec_ollama(
            api_endpoint="http://ollama:11434",
            model="nomic-embed-text"
        ),
        properties=[
            wvc.config.Property(name="product_id", data_type=wvc.config.DataType.INT)
        ]
    )

    with products.batch.fixed_size(batch_size=100) as batch:
        for idx, row in prod_ext_df.iterrows():
            rec = row[DATA_FIELDS].to_dict()
            batch.add_object(properties=rec)


    

In [116]:
from weaviate.classes.query import Filter

with weaviate.connect_to_local() as client:
    products = client.collections.use("products")

    data = products.query.fetch_objects(include_vector=True)

    responses = products.query.near_text(
        query="",
        filters=Filter.by_property("category").equal("footwear") ,
        limit=5
    )


In [117]:

def filter_products(
    query: str = "",
    categories: list[str] | None = None,
    subcategories: list[str] | None = None,
    min_price: float | None = None,
    max_price: float | None = None,
    limit: int = 5,
):
    """Filter products based on query, category, subcategory, and price range."""

    filters = []

    if categories:
        filters.append(
            Filter.by_property("category").contains_any(categories)
        )

    if subcategories:
        filters.append(
            Filter.by_property("subcategory").contains_any(subcategories)
        )

    if min_price:
        filters.append(
            Filter.by_property("price").greater_or_equal(min_price)
        )

    if max_price:
        filters.append(
            Filter.by_property("price").less_or_equal(max_price)
        )

    

    filter_obj = functools.reduce(
        lambda x, y: x & y,
        filters
    )

    with weaviate.connect_to_local() as client:
        products = client.collections.use("products")

        responses = products.query.near_text(
            query=query,
            filters=filter_obj,
            limit=limit
        )

    return responses
    

In [118]:
responses = filter_products(
    query="black",
    categories=["footwear"],
    subcategories=["shoes"],
    min_price=0,
    max_price=100,
    limit=5,
)

In [119]:
for obj in responses.objects:
    print(json.dumps(obj.properties, indent=2))

{
  "description": "Introducing the Flat Strappy Black Sandals with Buckel Embellishment, a blend of style and comfort designed to enhance your casual and special occasion attire. These sandals boast a minimalist yet sophisticated design, featuring thin black straps that delicately crisscross over the foot. At the toe, a subtle yet eye-catching metal embellishment adds a touch of elegance and refinement. The black straps create a clean and versatile look, ensuring these sandals pair effortlessly with a range of outfits from jeans and a blouse to summer dresses.\nWith a sole height of 0.6 inches (1.5 cm), these sandals offer a comfortable fit suitable for prolonged wear. The buckle closure provides a secure, adjustable fit. Whether you're walking through a museum or lounging at a street cafe, these sandals promise stability and ease.\nConstructed from durable materials, the straps are made from sturdy and flexible materials that conform comfortably to your feet, ensuring a snug and supp

In [92]:
with weaviate.connect_to_local() as client:
    print(client.collections.list_all())

{'Products': _CollectionConfigSimple(name='Products', description=None, generative_config=None, properties=[_Property(name='product_id', description="This property was generated by Weaviate's auto-schema feature on Tue Nov 25 07:15:57 2025", data_type=<DataType.NUMBER: 'number'>, index_filterable=True, index_range_filters=False, index_searchable=False, nested_properties=None, tokenization=None, vectorizer_config=None, vectorizer=None, vectorizer_configs={'text2vec-ollama': _PropertyVectorizerConfig(skip=False, vectorize_property_name=False)}), _Property(name='category', description="This property was generated by Weaviate's auto-schema feature on Tue Nov 25 07:15:57 2025", data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=None, vectorizer=None, vectorizer_configs={'text2vec-ollama': _PropertyVectorizerConfig(skip=False, vectorize_property_name=Fa